# Acquisition metadata (collision energy, instrument) for the library entries

Builds the USI of every scan contributing to a library entry, merges the scan-level acquisition metadata retrieved for those USIs (`bile_acid_scan_metadata.tsv`) and adds it to the final library annotation tables deposited in Zenodo.

In [ ]:
import os

# Root of the local project folder (the one holding Falcon/, Queries_final_mgf/,
# final_mgfs_library_generation/, ...). Edit this single line to run elsewhere.
PROJECT_DIR = '/media/dorrestein/Helena2/Bile_acids_update'

In [ ]:
import pandas as pd

In [ ]:
c23 = pd.read_csv(f'{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/TSV_and_HTML_files_artifacts/cores_combined_6_total/C23_combined_annotated_artifacts.tsv', sep='\t')
c24 = pd.read_csv(f'{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/TSV_and_HTML_files_artifacts/cores_combined_6_total/C24_combined_annotated_artifacts.tsv', sep='\t')
c24_alkamine = pd.read_csv(f'{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/TSV_and_HTML_files_artifacts/cores_combined_6_total/C24_alkylamine_combined_annotated_artifacts.tsv', sep='\t')
c26 = pd.read_csv(f'{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/TSV_and_HTML_files_artifacts/cores_combined_6_total/Aceto_combined_annotated_artifacts.tsv', sep='\t')
c27 = pd.read_csv(f'{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/TSV_and_HTML_files_artifacts/cores_combined_6_total/C27_combined_annotated_artifacts.tsv', sep='\t')
c28_alcohol = pd.read_csv(f'{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/TSV_and_HTML_files_artifacts/cores_combined_6_total/C28_alcohol_combined_annotated_artifacts.tsv', sep='\t')

#filter for only columns of interest and create a new column for the core type
c23 = c23[['original_path_scan', 'identifier']]
c23['BA_core'] = 'C23'

c24 = c24[['original_path_scan', 'identifier']]
c24['BA_core'] = 'C24'

c24_alkamine = c24_alkamine[['original_path_scan', 'identifier']]
c24_alkamine['BA_core'] = 'C24_alkylamine'

c26 = c26[['original_path_scan', 'identifier']]
c26['BA_core'] = 'C26'

c27 = c27[['original_path_scan', 'identifier']]
c27['BA_core'] = 'C27'

c28_alcohol = c28_alcohol[['original_path_scan', 'identifier']]
c28_alcohol['BA_core'] = 'C28_alcohol'

In [ ]:
c23_library = pd.read_csv(f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/C23_annotation_combined_scan.tsv', sep='\t')
c24_library = pd.read_csv(f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/C24_annotation_combined_scan.tsv', sep='\t')
c24_alkamine_library = pd.read_csv(f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/C24_alkylamine_annotation_combined_scan.tsv', sep='\t')
c26_library = pd.read_csv(f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/C26_annotation_combined_scan.tsv', sep='\t')
c27_library = pd.read_csv(f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/C27_annotation_combined_scan.tsv', sep='\t')
c28_alcohol_library = pd.read_csv(f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/C28_alcohol_annotation_combined_scan.tsv', sep='\t')

In [ ]:
c23_combined = pd.merge(c23_library, c23, on='identifier', how='left')
c24_combined = pd.merge(c24_library, c24, on='identifier', how='left')
c24_alkamine_combined = pd.merge(c24_alkamine_library, c24_alkamine, on='identifier', how='left')
c26_combined = pd.merge(c26_library, c26, on='identifier', how='left')
c27_combined = pd.merge(c27_library, c27, on='identifier', how='left')
c28_alcohol_combined = pd.merge(c28_alcohol_library, c28_alcohol, on='identifier', how='left')

In [ ]:
#split in _ and keep all string except for the last one for filepath
#split in _ and keep the last one for scan number
for df in [c23_combined, c24_combined, c24_alkamine_combined, c26_combined, c27_combined, c28_alcohol_combined]:
    df['original_path'] = df['original_path_scan'].str.rsplit('_', n=1).str[0]
    df['scan_number'] = df['original_path_scan'].str.rsplit('_', n=1).str[-1]

In [ ]:
#concatenate all of them vertically
all_combined = pd.concat([c23_combined, c24_combined, c24_alkamine_combined, c26_combined, c27_combined, c28_alcohol_combined], axis=0)

In [ ]:
#put in the form of a universal spectrum identifier (USI) for GNPS
# with mzspec: + dataset + : + filepath + : + scan + : scan number

all_combined['dataset'] = all_combined['original_path'].str.split('/').str[2]
#filepath, split in / and keep everything after the 3rd element (index 2)
all_combined['filepath'] = all_combined['original_path'].str.split('/').str[3:].str.join('/')
all_combined['USI'] = 'mzspec:' + all_combined['dataset'].astype(str) + ':' + all_combined['filepath'].astype(str) + ':' + 'scan' + ':' + all_combined['scan_number'].astype(str)
all_combined['MRI'] = 'mzspec:' + all_combined['dataset'].astype(str) + ':' + all_combined['filepath'].astype(str)

# all_combined = all_combined[['dataset', 'filepath', 'scan_number', 'USI', 'MRI']]

In [ ]:
all_combined.to_csv(f'{PROJECT_DIR}/Scripts_data_analysis/collision_energy/list_USIs_bile_acid_update_library_ok.tsv', sep='\t', index=False)

## Michael's workflow results

In [ ]:
#import
df = pd.read_csv(f'{PROJECT_DIR}/Scripts_data_analysis/collision_energy/bile_acid_scan_metadata.tsv', sep='\t')

#rename columns
df = df.rename(columns={'precursor_collision_energy': 'COLLISION_ENERGY', 
                        'analyzer': 'ANALYZER',
                        'ionization': 'IONIZATION',
                        'instrument_vendor': 'INSTRUMENT_VENDOR',
                        'instrument_model': 'INSTRUMENT_MODEL',
                        'precursor_activation': 'PRECURSOR_ACTIVATION',})

In [ ]:
## Add the information Michael provided to the all_combined dataframe
 
all_combined_merged = pd.merge(all_combined, df[['USI', 'PRECURSOR_ACTIVATION', 'COLLISION_ENERGY', 'ANALYZER', 'IONIZATION', 'INSTRUMENT_VENDOR', 'INSTRUMENT_MODEL']], on='USI', how='left')
all_combined_merged = all_combined_merged.drop_duplicates()

all_combined_merged.to_csv(f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/all_combined_collision_energy_merged.tsv', sep='\t', index=False)

In [ ]:
#import the library deposition files to include the additinoal columns

c23_library_deposition = pd.read_csv(f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/C23_annotation_combined_scan.tsv', sep='\t')
c24_library_deposition = pd.read_csv(f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/C24_annotation_combined_scan.tsv', sep='\t')
c24_alkamine_library_deposition = pd.read_csv(f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/C24_alkylamine_annotation_combined_scan.tsv', sep='\t')
c26_library_deposition = pd.read_csv(f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/C26_annotation_combined_scan.tsv', sep='\t')
c27_library_deposition = pd.read_csv(f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/C27_annotation_combined_scan.tsv', sep='\t')
c28_alcohol_library_deposition = pd.read_csv(f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/C28_alcohol_annotation_combined_scan.tsv', sep='\t')

In [ ]:
#merge
c23_library_deposition = pd.merge(c23_library_deposition, all_combined_merged[['identifier','USI', 'PRECURSOR_ACTIVATION', 'COLLISION_ENERGY', 'ANALYZER', 'IONIZATION', 'INSTRUMENT_VENDOR', 'INSTRUMENT_MODEL']], on='identifier', how='left').drop_duplicates()
c24_library_deposition = pd.merge(c24_library_deposition, all_combined_merged[['identifier', 'USI', 'PRECURSOR_ACTIVATION', 'COLLISION_ENERGY', 'ANALYZER', 'IONIZATION', 'INSTRUMENT_VENDOR', 'INSTRUMENT_MODEL']], on='identifier', how='left').drop_duplicates()
c24_alkamine_library_deposition = pd.merge(c24_alkamine_library_deposition, all_combined_merged[['identifier', 'USI', 'PRECURSOR_ACTIVATION', 'COLLISION_ENERGY', 'ANALYZER', 'IONIZATION', 'INSTRUMENT_VENDOR', 'INSTRUMENT_MODEL']], on='identifier', how='left').drop_duplicates()
c26_library_deposition = pd.merge(c26_library_deposition, all_combined_merged[['identifier', 'USI', 'PRECURSOR_ACTIVATION', 'COLLISION_ENERGY', 'ANALYZER', 'IONIZATION', 'INSTRUMENT_VENDOR', 'INSTRUMENT_MODEL']], on='identifier', how='left').drop_duplicates()
c27_library_deposition = pd.merge(c27_library_deposition, all_combined_merged[['identifier', 'USI', 'PRECURSOR_ACTIVATION', 'COLLISION_ENERGY', 'ANALYZER', 'IONIZATION', 'INSTRUMENT_VENDOR', 'INSTRUMENT_MODEL']], on='identifier', how='left').drop_duplicates()
c28_alcohol_library_deposition = pd.merge(c28_alcohol_library_deposition, all_combined_merged[['identifier', 'USI', 'PRECURSOR_ACTIVATION', 'COLLISION_ENERGY', 'ANALYZER', 'IONIZATION', 'INSTRUMENT_VENDOR', 'INSTRUMENT_MODEL']], on='identifier', how='left').drop_duplicates()

In [ ]:
c23_final = pd.read_csv(f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/final_files_GNPS/C23_library_final_annotation_artifacts_lib_match.tsv', sep='\t')
c24_final = pd.read_csv(f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/final_files_GNPS/C24_library_final_annotation_artifacts_lib_match.tsv', sep='\t')
c24_alkamine_final = pd.read_csv(f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/final_files_GNPS/C24_alkylamine_library_final_annotation_artifacts_lib_match.tsv', sep='\t')
c26_final = pd.read_csv(f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/final_files_GNPS/C26_library_final_annotation_artifacts_lib_match.tsv', sep='\t')
c27_final = pd.read_csv(f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/final_files_GNPS/C27_library_final_annotation_artifacts_lib_match.tsv', sep='\t')
c28_alcohol_final = pd.read_csv(f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/final_files_GNPS/C28_alcohol_library_final_annotation_artifacts_lib_match.tsv', sep='\t')

In [ ]:
#merge in the deposition table

c23_library_deposition = c23_library_deposition.rename(columns={'scan_final_mgf': 'EXTRACTSCAN'})
c23_final_merged = pd.merge(c23_final, c23_library_deposition[['EXTRACTSCAN', 'USI', 'PRECURSOR_ACTIVATION', 'COLLISION_ENERGY', 'ANALYZER', 'IONIZATION', 'INSTRUMENT_VENDOR', 'INSTRUMENT_MODEL']], on='EXTRACTSCAN', how='left').drop_duplicates()

c24_library_deposition = c24_library_deposition.rename(columns={'scan_final_mgf': 'EXTRACTSCAN'})
c24_final_merged = pd.merge(c24_final, c24_library_deposition[['EXTRACTSCAN', 'USI', 'PRECURSOR_ACTIVATION', 'COLLISION_ENERGY', 'ANALYZER', 'IONIZATION', 'INSTRUMENT_VENDOR', 'INSTRUMENT_MODEL']], on='EXTRACTSCAN', how='left').drop_duplicates()

c24_alkamine_library_deposition = c24_alkamine_library_deposition.rename(columns={'scan_final_mgf': 'EXTRACTSCAN'})
c24_alkamine_final_merged = pd.merge(c24_alkamine_final, c24_alkamine_library_deposition[['EXTRACTSCAN', 'USI', 'PRECURSOR_ACTIVATION', 'COLLISION_ENERGY', 'ANALYZER', 'IONIZATION', 'INSTRUMENT_VENDOR', 'INSTRUMENT_MODEL']], on='EXTRACTSCAN', how='left').drop_duplicates()

c26_library_deposition = c26_library_deposition.rename(columns={'scan_final_mgf': 'EXTRACTSCAN'})
c26_final_merged = pd.merge(c26_final, c26_library_deposition[['EXTRACTSCAN', 'USI', 'PRECURSOR_ACTIVATION', 'COLLISION_ENERGY', 'ANALYZER', 'IONIZATION', 'INSTRUMENT_VENDOR', 'INSTRUMENT_MODEL']], on='EXTRACTSCAN', how='left').drop_duplicates()

c27_library_deposition = c27_library_deposition.rename(columns={'scan_final_mgf': 'EXTRACTSCAN'})
c27_final_merged = pd.merge(c27_final, c27_library_deposition[['EXTRACTSCAN', 'USI', 'PRECURSOR_ACTIVATION', 'COLLISION_ENERGY', 'ANALYZER', 'IONIZATION', 'INSTRUMENT_VENDOR', 'INSTRUMENT_MODEL']], on='EXTRACTSCAN', how='left').drop_duplicates()

c28_alcohol_library_deposition = c28_alcohol_library_deposition.rename(columns={'scan_final_mgf': 'EXTRACTSCAN'})
c28_alcohol_final_merged = pd.merge(c28_alcohol_final, c28_alcohol_library_deposition[['EXTRACTSCAN', 'USI', 'PRECURSOR_ACTIVATION', 'COLLISION_ENERGY', 'ANALYZER', 'IONIZATION', 'INSTRUMENT_VENDOR', 'INSTRUMENT_MODEL']], on='EXTRACTSCAN', how='left').drop_duplicates()

In [ ]:
#export
c23_final_merged.to_csv(f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/final_files_GNPS/C23_library_final_annotation_artifacts_lib_match_CE_final.tsv', sep='\t', index=False)
c24_final_merged.to_csv(f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/final_files_GNPS/C24_library_final_annotation_artifacts_lib_match_CE_final.tsv', sep='\t', index=False)
c24_alkamine_final_merged.to_csv(f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/final_files_GNPS/C24_alkylamine_library_final_annotation_artifacts_lib_match_CE_final.tsv', sep='\t', index=False)
c26_final_merged.to_csv(f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/final_files_GNPS/C26_library_final_annotation_artifacts_lib_match_CE_final.tsv', sep='\t', index=False)
c27_final_merged.to_csv(f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/final_files_GNPS/C27_library_final_annotation_artifacts_lib_match_CE_final.tsv', sep='\t', index=False)
c28_alcohol_final_merged.to_csv(f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/final_files_GNPS/C28_alcohol_library_final_annotation_artifacts_lib_match_CE_final.tsv', sep='\t', index=False)